# Avaliação de dificuldade — questões da Fase 3

Este notebook estima a dificuldade **intrínseca** de cada questão de múltipla escolha da Fase 3 usando o backend Azure OpenAI já disponível no projeto. A avaliação considera o que a questão exige de um candidato bem preparado; não mede o desempenho de um modelo respondendo à questão.

A rubrica foi adaptada de `agents/difficulty_verification.py` e `prompts/difficulty_verification.py`: integração de conhecimento, cadeia de raciocínio, demanda cognitiva, densidade/especificidade técnica e plausibilidade dos distratores. Como o arquivo final da Fase 3 não conserva os trechos-fonte nem o plano pedagógico do outro projeto, estes não são usados.

## Persistência e retomada

As questões são enviadas em lotes pequenos. Logo após cada resposta, cada questão do lote é escrita em `resultados_dificuldade_azure/avaliacoes_checkpoint.jsonl`. Se a execução parar, reexecute a célula de avaliação: somente questões sem resultado válido serão reenviadas. Ao final, há uma célula que gera o CSV consolidado.

> O notebook não altera o banco de questões original.

In [ ]:
import json
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


def find_project_root(start: Path) -> Path:
    """Localiza a raiz mesmo se o Jupyter tiver sido iniciado em outra pasta."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'azure_openai_backend.py').is_file():
            return candidate
    raise RuntimeError('Não encontrei azure_openai_backend.py acima do diretório atual.')


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from azure_openai_backend import AzureOpenAIBackend

INPUT_PATH = ROOT / 'pipeline' / 'fase_3' / 'saida_fase3' / 'questoes_fase3.jsonl'
OUT_DIR = ROOT / 'pipeline' / 'fase_4' / 'resultados_dificuldade_azure'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUT_DIR / 'avaliacoes_checkpoint.jsonl'
CSV_PATH = OUT_DIR / 'dificuldades_fase3.csv'

# Deployment Azure. Para uma rodada mais econômica, use, por exemplo,
# DEPLOYMENT = 'gpt-5-4-mini-petrobras' (se este for o nome disponível no gateway).
DEPLOYMENT = 'gpt-5-4-petrobras'
BATCH_SIZE = 5
EVALUATOR_VERSION = 'difficulty-v1-final-question'

pd.set_option('display.max_colwidth', 160)
print(f'Raiz do projeto: {ROOT}')
print(f'Entrada: {INPUT_PATH}')
print(f'Checkpoint: {CHECKPOINT_PATH}')

## Carregar e validar a entrada

O padrão é `questoes_fase3.jsonl`, o conjunto final com 493 questões. Para avaliar o repositório histórico completo, substitua `INPUT_PATH` na célula anterior por `.../repositorio/questoes.jsonl` e use outro `EVALUATOR_VERSION` para não misturar resultados.

In [ ]:
def load_questions(path: Path) -> list[dict]:
    if not path.is_file():
        raise FileNotFoundError(f'Arquivo de questões não encontrado: {path}')

    questions, seen_ids = [], set()
    required = {'id', 'stem', 'alternatives', 'correct_answer_index'}
    with path.open(encoding='utf-8') as handle:
        for line_no, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            question = json.loads(line)
            missing = required - question.keys()
            if missing:
                raise ValueError(f'Linha {line_no}: campos ausentes: {sorted(missing)}')
            if question['id'] in seen_ids:
                raise ValueError(f'ID duplicado na linha {line_no}: {question["id"]}')
            if not 2 <= len(question['alternatives']) <= 6:
                raise ValueError(f'{question["id"]}: esperado de 2 a 6 alternativas.')
            if not 0 <= question['correct_answer_index'] < len(question['alternatives']):
                raise ValueError(f'{question["id"]}: correct_answer_index inválido.')
            seen_ids.add(question['id'])
            questions.append(question)
    return questions


questions = load_questions(INPUT_PATH)
print(f'{len(questions)} questões carregadas.')
pd.DataFrame([{
    'id': q['id'], 'topico': q.get('topico'), 'difficulty_original': q.get('difficulty'),
    'n_alternatives': len(q['alternatives']), 'stem': q['stem']
} for q in questions]).head()

## Rubrica e conexão com o Azure

A nota final é a média simples de cinco dimensões, de 1 a 3: `1,00–1,66 = fácil`, `1,67–2,33 = média` e `2,34–3,00 = difícil`. A resposta correta é fornecida ao avaliador apenas para ele julgar a qualidade diagnóstica dos distratores; ela não é exibida na saída destinada ao candidato.

O backend já cuida de credenciais, certificado corporativo, repetição de chamadas transitórias e cache. Em modelos de raciocínio da família GPT-5, ele também usa o orçamento de tokens compatível com esse tipo de deployment.

In [ ]:
SYSTEM_PROMPT = '''Você é um especialista em avaliação educacional técnica.
Sua tarefa é estimar a dificuldade intrínseca de questões de múltipla escolha para o público
que estudou adequadamente o domínio. Julgue o que é necessário para chegar à resposta correta,
não a qualidade da redação nem a dificuldade aparente causada por vocabulário longo.

Use exatamente estas cinco dimensões, sempre com inteiros de 1 a 3:
- integracao_conhecimento: 1=recordação/localização direta de um conceito; 2=interpretação ou aplicação de um conceito em contexto; 3=integração de dois ou mais conceitos/condições para decidir.
- cadeia_raciocinio: 1=uma decisão direta; 2=comparação, relação causal ou aplicação em mais de uma etapa; 3=sequência indispensável de múltiplas inferências ou trade-offs.
- demanda_cognitiva: 1=lembrar/compreender; 2=aplicar; 3=analisar, avaliar ou escolher sob critérios concorrentes.
- especificidade_tecnica: 1=conhecimento geral ou definição canônica; 2=princípio técnico contextualizado; 3=conhecimento técnico especializado, condicionado ou quantitativo.
- plausibilidade_distratores: 1=dois ou mais distratores são elimináveis sem resolver; 2=um distrator é eliminável cedo, mas os demais competem; 3=nenhum distrator pode ser eliminado antes da resolução completa.

Ignore o campo de dificuldade original. Não reavalie a veracidade factual da chave fornecida;
presuma que ela é o gabarito editorial. Para cada item, produza justificativas curtas, concretas
e em português. Retorne exclusivamente JSON válido no formato solicitado.'''


def question_for_evaluation(question: dict) -> dict:
    letters = 'ABCDEF'
    return {
        'id': question['id'],
        'topico': question.get('topico'),
        'subtopico': question.get('subtopico'),
        'enunciado': question['stem'],
        'alternativas': {letters[i]: text for i, text in enumerate(question['alternatives'])},
        'gabarito_editorial': letters[question['correct_answer_index']],
    }


def build_batch_prompt(batch: list[dict]) -> str:
    payload = [question_for_evaluation(question) for question in batch]
    return (
        'Avalie cada questão abaixo de forma independente.\n\n'
        'QUESTÕES:\n' + json.dumps(payload, ensure_ascii=False, indent=2) + '\n\n'
        'Retorne somente este objeto JSON, sem Markdown:\n'
        '{"avaliacoes": [{"id": "...", "integracao_conhecimento": 1, '
        '"cadeia_raciocinio": 1, "demanda_cognitiva": 1, '
        '"especificidade_tecnica": 1, "plausibilidade_distratores": 1, '
        '"justificativa": "...", "confianca": "alta|media|baixa"}]}\n'
        'Inclua exatamente uma avaliação para cada ID recebido.'
    )


backend = AzureOpenAIBackend(
    deployment=DEPLOYMENT,
    max_tokens=6000,
    reasoning_min_tokens=6000,
    reasoning_effort='low',
    request_timeout=180,
    cache_dir=str(OUT_DIR / 'cache_azure'),
)
print(f'Deployment: {backend.deployment} | modelo de raciocínio: {backend.reasoning}')

## Avaliar em lotes, com checkpoint por questão

Cada lote gera uma chamada ao Azure, mas a persistência é individual. Resultados parciais, JSON inválido ou falhas ficam registrados com `status=erro` e permanecem pendentes para a próxima execução.

In [ ]:
DIMENSIONS = (
    'integracao_conhecimento', 'cadeia_raciocinio', 'demanda_cognitiva',
    'especificidade_tecnica', 'plausibilidade_distratores',
)


def load_completed_ids(path: Path) -> dict[str, dict]:
    """Usa o último registro válido de cada questão nesta versão do avaliador."""
    completed = {}
    if not path.exists():
        return completed
    with path.open(encoding='utf-8') as handle:
        for raw_line in handle:
            if not raw_line.strip():
                continue
            try:
                record = json.loads(raw_line)
            except json.JSONDecodeError:
                continue
            if (record.get('evaluator_version') == EVALUATOR_VERSION
                    and record.get('deployment') == DEPLOYMENT
                    and record.get('status') == 'ok'):
                completed[record['id']] = record
    return completed


def validate_evaluation(item: dict, expected_id: str) -> dict:
    if not isinstance(item, dict) or item.get('id') != expected_id:
        raise ValueError('ID ausente ou diferente do esperado.')
    clean = {'id': expected_id}
    for dimension in DIMENSIONS:
        value = item.get(dimension)
        if isinstance(value, bool) or not isinstance(value, int) or value not in (1, 2, 3):
            raise ValueError(f'{dimension} precisa ser inteiro entre 1 e 3.')
        clean[dimension] = value
    justification = item.get('justificativa')
    if not isinstance(justification, str) or not justification.strip():
        raise ValueError('justificativa ausente.')
    clean['justificativa'] = justification.strip()
    confidence = str(item.get('confianca', 'media')).lower()
    clean['confianca'] = confidence if confidence in {'alta', 'media', 'baixa'} else 'media'
    score = round(sum(clean[d] for d in DIMENSIONS) / len(DIMENSIONS), 2)
    clean['difficulty_score'] = score
    clean['difficulty_label'] = 'facil' if score <= 1.66 else ('media' if score <= 2.33 else 'dificil')
    return clean


def chunks(items: list[dict], size: int):
    for start in range(0, len(items), size):
        yield items[start:start + size]


completed = load_completed_ids(CHECKPOINT_PATH)
pending = [question for question in questions if question['id'] not in completed]
print(f'{len(completed)} questões concluídas; {len(pending)} pendentes.')

with CHECKPOINT_PATH.open('a', encoding='utf-8') as checkpoint:
    for batch in tqdm(list(chunks(pending, BATCH_SIZE)), desc='Lotes Azure'):
        timestamp = datetime.now(timezone.utc).isoformat()
        try:
            response = backend.complete_json(
                build_batch_prompt(batch), system=SYSTEM_PROMPT, max_tokens=6000
            )
            by_id = {item.get('id'): item for item in response.get('avaliacoes', []) if isinstance(item, dict)}
            if not isinstance(response, dict):
                raise ValueError('A resposta não é um objeto JSON.')
        except Exception as exc:  # O backend já tenta novamente falhas transitórias.
            by_id = {}
            batch_error = f'{type(exc).__name__}: {exc}'
        else:
            batch_error = None

        for question in batch:
            base = {
                'id': question['id'], 'deployment': DEPLOYMENT,
                'evaluator_version': EVALUATOR_VERSION, 'evaluated_at': timestamp,
            }
            try:
                if batch_error:
                    raise RuntimeError(batch_error)
                record = base | validate_evaluation(by_id.get(question['id']), question['id'])
                record['status'] = 'ok'
                completed[question['id']] = record
            except Exception as exc:
                record = base | {'status': 'erro', 'error': f'{type(exc).__name__}: {exc}'}
            checkpoint.write(json.dumps(record, ensure_ascii=False) + '\n')
            checkpoint.flush()

print(f'Checkpoint atualizado: {CHECKPOINT_PATH}')
print(f'Chamadas: {backend.usage.calls}; cache: {backend.usage.cached_calls}; tokens: {backend.usage.total_tokens}')

## Consolidar e exportar

A tabela preserva os metadados da Fase 3 e adiciona a nota, a classificação e as cinco dimensões. Só entram no CSV as avaliações concluídas. Execute a célula após a rodada — inclusive para obter um CSV parcial durante a execução.

In [ ]:
completed = load_completed_ids(CHECKPOINT_PATH)
rows = []
for question in questions:
    evaluation = completed.get(question['id'])
    if not evaluation:
        continue
    rows.append({
        'id': question['id'],
        'topico': question.get('topico'),
        'subtopico': question.get('subtopico'),
        'faceta_titulo': question.get('faceta_titulo'),
        'difficulty_original': question.get('difficulty'),
        'difficulty_score': evaluation['difficulty_score'],
        'difficulty_label': evaluation['difficulty_label'],
        'confianca': evaluation['confianca'],
        **{dimension: evaluation[dimension] for dimension in DIMENSIONS},
        'justificativa': evaluation['justificativa'],
        'deployment': evaluation['deployment'],
        'evaluator_version': evaluation['evaluator_version'],
        'evaluated_at': evaluation['evaluated_at'],
        'stem': question['stem'],
    })

if rows:
    results = pd.DataFrame(rows).sort_values(['difficulty_score', 'id'], ascending=[False, True])
else:
    results = pd.DataFrame(columns=['id', 'difficulty_score', 'difficulty_label'])
results.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
print(f'{len(results)} de {len(questions)} questões exportadas em: {CSV_PATH}')
if len(results):
    display(results['difficulty_label'].value_counts().rename_axis('dificuldade').to_frame('quantidade'))
    display(results.head(10))